# 📈 Stock Data Analysis

這個 Notebook 專門用來調用 `database` 模組中的資料，並將其轉換為 Pandas DataFrame 以供分析。

### 前置準備
請確保您已安裝以下套件：
```bash
pip install pandas plotly jupyterlab
```

In [ ]:
import pandas as pd
import sys
import os

# 1. 設定路徑：確保可以 import 專案根目錄的模組
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.append(current_dir)

try:
    from database.client import get_db_client
    print("✅ 成功載入 database 模組")
except ImportError as e:
    print(f"❌ 載入失敗: {e}")
    print("   請確認 database/client.py 存在且路徑正確。")

In [ ]:
def get_table_as_df(table_name):
    """
    使用專案的 db client 讀取整張資料表
    """
    print(f"📥 正在讀取資料表: {table_name} ...")
    try:
        with get_db_client() as db:
            # 根據 crawlers/price.py 的用法，db.fetch 回傳 DataFrame
            df = db.fetch(table_name)
            print(f"✅ 讀取完成！共 {len(df)} 筆資料")
            return df
    except Exception as e:
        print(f"❌ 讀取錯誤: {e}")
        return pd.DataFrame()

## 1. 讀取台股資料 (TWStock)

In [ ]:
df_tw = get_table_as_df("tw_daily_prices")

# 預覽前 5 筆
df_tw.head()

## 2. 讀取美股資料 (USStock)

In [ ]:
df_us = get_table_as_df("us_daily_prices")

# 預覽前 5 筆
df_us.head()

## 3. 簡單視覺化範例 (Plotly)
這裡示範如何畫出 K 線圖。

In [ ]:
import plotly.graph_objects as go

def plot_stock(df, stock_id_or_ticker):
    if df.empty:
        print("⚠️ 資料集為空，無法繪圖")
        return
    
    # 判斷欄位名稱 (台股用 stock_id, 美股用 ticker)
    col_name = 'stock_id' if 'stock_id' in df.columns else 'ticker'
    
    # 篩選資料
    target_df = df[df[col_name] == stock_id_or_ticker].copy()
    
    if target_df.empty:
        print(f"⚠️ 找不到代號 {stock_id_or_ticker} 的資料")
        return
        
    # 確保日期格式
    target_df['date'] = pd.to_datetime(target_df['date'])
    target_df = target_df.sort_values('date')
    
    # 繪圖
    fig = go.Figure(data=[go.Candlestick(
        x=target_df['date'],
        open=target_df['open'],
        high=target_df['high'],
        low=target_df['low'],
        close=target_df['close'],
        name=stock_id_or_ticker
    )])
    
    fig.update_layout(
        title=f"{stock_id_or_ticker} 股價走勢",
        yaxis_title="Price",
        xaxis_title="Date",
        template="plotly_dark"
    )
    fig.show()

# 測試：畫出台積電 (2330) 或其他存在的股票
# 請確保資料庫內有該股票資料
plot_stock(df_tw, "2330")

## 4. 計算技術指標 (Technical Indicators)
這裡我們將引用 `indicators.py` 中的邏輯，自動計算均線 (MA)、乖離率等指標。

In [ ]:
try:
    from indicators import calculate_technical_indicators
    print("✅ 成功載入 indicators 模組")
except ImportError:
    print("❌ 找不到 indicators.py，請確認檔案位於專案根目錄")

# 確保 df_tw 已經讀取 (如果前面沒跑，這裡重讀一次)
if 'df_tw' not in locals() or df_tw.empty:
    df_tw = get_table_as_df("tw_daily_prices")

if not df_tw.empty:
    # 🔥 修改點：不再只篩選一檔，而是對「全市場」股票進行計算
    print(f"📊 正在為全市場 {df_tw['stock_id'].nunique()} 檔股票計算技術指標 (這可能需要一點時間)...")
    
    # indicators.py 裡的邏輯已經支援 groupby('stock_id')，所以可以直接丟整個 df 進去
    df_all_indicators = calculate_technical_indicators(df_tw)
    print("✅ 計算完成！")
    
    # --- 範例應用：找出強勢股 ---
    # 1. 取出最近一天的資料
    latest_date = df_all_indicators['date'].max()
    df_today = df_all_indicators[df_all_indicators['date'] == latest_date].copy()
    
    print(f"\n🏆 {latest_date} 市場掃描結果:")
    
    # 2. 策略 A: 量能爆發 (含防妖濾網)
    # 計算 20日均成交值 (億) - 近似值: 20日均量 * 目前股價
    df_today['avg_turnover_20d'] = (df_today['avg_vol_baseline'] * df_today['close']) / 100000000
    
    # 防妖濾網: 均量>2000張, 均值>1億, 股價>20元, 多頭排列
    mask = (
        (df_today['close'] > df_today['price_ma20']) & 
        (df_today['vol_ratio'] > 2.0) & 
        (df_today['avg_vol_baseline'] > 2000000) & 
        (df_today['avg_turnover_20d'] > 1.0) & 
        (df_today['close'] > 20)
    )
    
    top_volume = df_today[mask].sort_values('vol_ratio', ascending=False).head(5)
    print("\n🛡️ [防妖濾網] 量能爆發且流動性佳 Top 5:")
    display(top_volume[['stock_id', 'stock_name', 'close', 'vol_ratio', 'avg_turnover_20d', 'bias']])
    
    # 3. 策略 B: 乖離率過大 (正乖離 > 10%，可能過熱)
    overheated = df_today[df_today['bias'] > 0.1].sort_values('bias', ascending=False).head(5)
    print("\n⚠️ 短線過熱 (乖離率 > 10%) Top 5:")
    display(overheated[['stock_id', 'stock_name', 'close', 'bias', 'volatility']])
    
else:
    print("⚠️ 找不到資料，請先確認資料庫是否有該股票數據")

In [ ]:
# (進階) 畫出包含均線的 K 線圖
import plotly.graph_objects as go

# 設定你想畫的股票代號
target_stock_id = "2330"  # 可以改成上面掃描出來的強勢股代號

if 'df_all_indicators' in locals() and not df_all_indicators.empty:
    # 從計算好的大表中篩選出這一檔
    df_plot = df_all_indicators[df_all_indicators['stock_id'] == target_stock_id].copy()
    
    if not df_plot.empty:
        fig = go.Figure()

        # K線
        fig.add_trace(go.Candlestick(
            x=df_plot['date'],
            open=df_plot['open'], high=df_plot['high'],
            low=df_plot['low'], close=df_plot['close'],
            name='K線'
        ))

        # MA5 (黃線)
        fig.add_trace(go.Scatter(
            x=df_plot['date'], y=df_plot['price_ma5'],
            mode='lines', name='MA5', line=dict(color='orange', width=1)
        ))

        # MA20 (紫線)
        fig.add_trace(go.Scatter(
            x=df_plot['date'], y=df_plot['price_ma20'],
            mode='lines', name='MA20', line=dict(color='purple', width=1)
        ))

        fig.update_layout(
            title=f"{target_stock_id} 技術分析圖 (含均線)",
            yaxis_title="Price",
            template="plotly_dark",
            height=600
        )
        fig.show()
    else:
        print(f"⚠️ 找不到 {target_stock_id} 的資料")
else:
    print("⚠️ 請先執行上一個 Cell 計算技術指標")